# VisionFlow Benchmarks (Kaggle)

Settings: **Accelerator → GPU T4 x2**, **Internet → On**. Add your test video as an input dataset before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf visionflow && git clone -q https://github.com/Altair098/visionflow.git
%cd /kaggle/working/visionflow
!pip install -q -r requirements.txt tensorrt

In [ ]:
import glob

videos = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True) + glob.glob("/kaggle/input/**/*.mov", recursive=True))
assert videos, "No video found. Add your video via 'Add Input' -> Upload dataset."
VIDEO = videos[0]
print(VIDEO)

In [ ]:
%%bash
mkdir -p /kaggle/working/datasets && cd /kaggle/working/datasets
if [ ! -d coco/images/val2017 ]; then
  wget -q https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017labels.zip
  unzip -q coco2017labels.zip && rm coco2017labels.zip
  rm -rf coco/labels/train2017 coco/train2017.txt
  mkdir -p coco/images && cd coco/images
  wget -q http://images.cocodataset.org/zips/val2017.zip
  unzip -q val2017.zip && rm val2017.zip
fi
ls /kaggle/working/datasets/coco/images/val2017 | wc -l

In [ ]:
import pathlib

import ultralytics
import yaml

coco = yaml.safe_load(open(pathlib.Path(ultralytics.__file__).parent / "cfg/datasets/coco.yaml"))
cfg = {"path": "/kaggle/working/datasets/coco", "train": "val2017.txt", "val": "val2017.txt", "names": coco["names"]}
yaml.safe_dump(cfg, open("coco_val.yaml", "w"))

In [ ]:
!python scripts/export.py --weights yolo11n.pt --format onnx
!python scripts/export.py --weights yolo11n.pt --format engine --half
!mv yolo11n.engine yolo11n_fp16.engine
!python scripts/export.py --weights yolo11n.pt --format engine --int8 --data coco_val.yaml
!mv yolo11n.engine yolo11n_int8.engine
!ls -lh yolo11n*

In [ ]:
import json
import subprocess

runs = [
    ("PyTorch", "FP32", "yolo11n.pt"),
    ("ONNX Runtime", "FP32", "yolo11n.onnx"),
    ("TensorRT", "FP16", "yolo11n_fp16.engine"),
    ("TensorRT", "INT8", "yolo11n_int8.engine"),
]

results = []
for backend, precision, weights in runs:
    proc = subprocess.run(
        ["python", "scripts/benchmark.py", "--weights", weights, "--source", VIDEO,
         "--frames", "500", "--data", "coco_val.yaml"],
        capture_output=True, text=True,
    )
    if proc.returncode != 0:
        print(proc.stderr[-2000:])
        continue
    r = json.loads(proc.stdout[proc.stdout.index("{"):])
    r.update(backend=backend, precision=precision)
    results.append(r)
    print(backend, precision, r, flush=True)

json.dump(results, open("/kaggle/working/single_stream.json", "w"), indent=2)

In [ ]:
!python scripts/multistream_benchmark.py --weights yolo11n_fp16.engine --source "{VIDEO}" --streams 1 2 4 8 12 16 --frames 300 --target-fps 15 --output /kaggle/working/multistream.json

In [ ]:
import json

results = json.load(open("/kaggle/working/single_stream.json"))
ms = json.load(open("/kaggle/working/multistream.json"))

base = results[0]["fps"]
print("| Model | Backend | Precision | FPS | Latency p50 (ms) | Latency p95 (ms) | mAP50-95 | Speedup |")
print("|---|---|---|---|---|---|---|---|")
for r in results:
    print(f"| YOLO11n | {r['backend']} | {r['precision']} | {r['fps']} | {r['latency_ms_p50']} | "
          f"{r['latency_ms_p95']} | {r['map50_95']} | {r['fps'] / base:.1f}x |")

print()
print("| Streams | Per-stream FPS | Total FPS | Real-time (>=15 FPS) |")
print("|---|---|---|---|")
for m in ms:
    print(f"| {m['streams']} | {m['per_stream_fps']} | {m['total_fps']} | {'yes' if m['realtime'] else 'no'} |")